# NexSafe AI — Kaggle GPU training

This notebook runs the repository's pinned, reproducible GPU pipeline. It fine-tunes YOLO11n on a deterministic **subset** of the real UVH-26 publisher train/validation splits (defaults: 400 / 100 images for a smoke run), saves a hash-bound NexSafe model sidecar, audits an attached ACCIDENT dataset if authorized, and can call the repository's temporal trainer only when separately curated, licensed, source-disjoint manifests contain genuine safe and accident labels. It does not generate negative labels or treat ACCIDENT-only accident clips as a safe-vs-accident dataset.

## Before running

1. In Kaggle, create a notebook with **GPU** accelerator and **Internet** enabled; restart the session after changing the accelerator.
2. Upload `D:\NexSafeAI\nexsafe-source-smoke-20261103-v3.zip` as a file in a private Kaggle Dataset named `nexsafe-source`, then add that dataset as an input. The notebook safely extracts its `backend/training` and `backend/app` source layout.
3. Read the current [ACCIDENT Kaggle dataset license](https://www.kaggle.com/datasets/picekl/accident). Its current API metadata reports **CC BY-NC-SA 4.0**, not a commercial-use license. You confirmed non-commercial research/share-alike use, which is recorded in the configuration cell. Do not use ACCIDENT or a model trained on it commercially without separate written permission. API acquisition remains off until you explicitly enable it and accept Kaggle's current dataset terms.
4. Review the official [Ultralytics license terms](https://www.ultralytics.com/license). AGPL-3.0 use requires compliance with its source-disclosure obligations; the current terms say commercial/private applications and SaaS/API deployments require an Enterprise License unless the project is fully open-sourced under AGPL. Because NexSafe is intended as a web platform, do not train or deploy the Ultralytics checkpoint for a proprietary/commercial service until you obtain the required license or confirm an AGPL-compliant open-source release. Leave `CONFIRM_ULTRALYTICS_LICENSE_REVIEWED = False` until you have resolved this.
5. Kaggle's current ACCIDENT archive is 20,158,641,093 bytes. Prefer adding it as a Kaggle input rather than extracting it into notebook working storage. API download is optional and has a conservative free-space check.
6. If Hugging Face returns HTTP 429, create a Kaggle Secret named `HF_TOKEN` with your own account token and uncomment the optional secure-loading lines in the configuration cell. Never put the token directly into notebook source.

This notebook was authored but **not executed** in the local CPU-only environment. No training results are implied.

In [ ]:
%pip install -q "huggingface_hub==0.36.2" "ultralytics==8.3.75"
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not active. Select a Kaggle GPU accelerator, restart the session, and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Kaggle working free GiB:", round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

input_root = Path('/kaggle/input')
source_archives = list(input_root.rglob('nexsafe-source*.zip'))
if len(source_archives) > 1:
    raise RuntimeError("Attach only one NexSafe source archive.")
if source_archives:
    extracted_source = Path('/kaggle/working/nexsafe-source')
    extracted_source.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_archives[0]) as archive:
        for member in archive.infolist():
            destination = (extracted_source / member.filename).resolve()
            if destination != extracted_source.resolve() and extracted_source.resolve() not in destination.parents:
                raise ValueError(f"Unsafe source archive path: {member.filename}")
        archive.extractall(extracted_source)
source_files = list(input_root.rglob('kaggle_gpu_pipeline.py')) + list(Path('/kaggle/working/nexsafe-source').rglob('kaggle_gpu_pipeline.py'))
candidates = {path.parent.parent.resolve() for path in source_files if path.parent.name == 'training'}
if len(candidates) != 1:
    raise FileNotFoundError("Attach one NexSafe source dataset/archive containing backend/training/kaggle_gpu_pipeline.py")
source_backend = candidates.pop()
print("NexSafe source:", source_backend)

## Configuration and secure data access

The Kaggle API key is never embedded in the notebook or printed. For API downloads, create Kaggle Secrets called `KAGGLE_USERNAME` and `KAGGLE_KEY`; alternatively, attach the authorized dataset through **Add Input** and leave API download off. The ACCIDENT non-commercial confirmation reflects your selection; the Ultralytics license confirmation remains false until you review those checkpoint terms.

In [ ]:
# Edit these only after reviewing the corresponding licenses/access terms.
CONFIRM_ULTRALYTICS_LICENSE_REVIEWED = False  # Keep false until the chosen AGPL/Enterprise path is verified.
ALLOW_NONCOMMERCIAL_ACCIDENT_USE = True  # User confirmed non-commercial research/share-alike use.
HAVE_SEPARATE_ACCIDENT_LICENSE = False
DOWNLOAD_ACCIDENT_VIA_KAGGLE_API = False

# A bounded real-data subset keeps Kaggle scratch storage manageable.
os.environ['NEXSAFE_MAX_TRAIN_IMAGES'] = '400'
os.environ['NEXSAFE_MAX_VAL_IMAGES'] = '100'
os.environ['NEXSAFE_EPOCHS'] = '5'
os.environ['NEXSAFE_BATCH'] = '4'
os.environ['NEXSAFE_IMAGE_SIZE'] = '640'
os.environ['NEXSAFE_SEED'] = '42'
os.environ['NEXSAFE_CONFIRM_ULTRALYTICS_LICENSE_REVIEWED'] = '1' if CONFIRM_ULTRALYTICS_LICENSE_REVIEWED else '0'
os.environ['NEXSAFE_ALLOW_NONCOMMERCIAL_ACCIDENT_USE'] = '1' if ALLOW_NONCOMMERCIAL_ACCIDENT_USE else '0'
os.environ['NEXSAFE_HAVE_SEPARATE_ACCIDENT_LICENSE'] = '1' if HAVE_SEPARATE_ACCIDENT_LICENSE else '0'
os.environ['NEXSAFE_DOWNLOAD_ACCIDENT'] = '1' if DOWNLOAD_ACCIDENT_VIA_KAGGLE_API else '0'
os.environ.setdefault('NEXSAFE_POSITIVE_LABELS', 'accident,near_miss')
os.environ.setdefault('NEXSAFE_NEGATIVE_LABELS', 'no_accident,non_accident,safe,normal,negative')

# Optional: after adding an HF_TOKEN Kaggle Secret, uncomment to authenticate downloads.
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

if DOWNLOAD_ACCIDENT_VIA_KAGGLE_API and not (ALLOW_NONCOMMERCIAL_ACCIDENT_USE or HAVE_SEPARATE_ACCIDENT_LICENSE):
    raise PermissionError("ACCIDENT download blocked: accept non-commercial/share-alike terms or obtain written permission first.")
if DOWNLOAD_ACCIDENT_VIA_KAGGLE_API and shutil.which('kaggle') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print("License gates configured; credentials will be read only if API download is enabled.")

## Run acquisition, audit, and GPU training

The pipeline downloads only the selected real UVH-26 files from the pinned Hugging Face revision into a configuration-keyed resumable cache under `/kaggle/working/nexsafe_training/cache/`. Re-running the same selection reuses completed downloads; other selections use separate caches. It verifies image dimensions against COCO labels, keeps the publisher's train/validation split, and does not invent a test split. It validates the saved best checkpoint, performs one actual inference pass on a validation image, and writes `uvh26-yolo11n.pt`, `uvh26-yolo11n.pt.nexsafe.json`, and an audit report under a timestamped `/kaggle/working/nexsafe_training/<run-id>/nexsafe_export/` directory.

If you attach ACCIDENT, the notebook reports its observed metadata columns, CSV labels and available video probes. The 2,027 real accident clips do not by themselves provide safe/negative examples; temporal training remains blocked unless you also attach separate, verified real-safe videos and owner-annotated, source-disjoint JSONL manifests. Synthetic and real runs must stay separate.

In [ ]:
pipeline = source_backend / 'training' / 'kaggle_gpu_pipeline.py'
environment = os.environ.copy()
environment['PYTHONPATH'] = str(source_backend) + os.pathsep + environment.get('PYTHONPATH', '')
result = subprocess.run(
    [sys.executable, str(pipeline)],
    cwd=str(source_backend),
    env=environment,
    check=False,
    capture_output=True,
    text=True,
)
print(result.stdout[-12000:])
if result.returncode:
    # Do not print environment variables or Kaggle secret values.
    print(result.stderr[-4000:])
    raise RuntimeError(f"NexSafe pipeline exited with code {result.returncode}")
run_line = next((line for line in result.stdout.splitlines() if line.startswith('Training run artifacts are under: ')), None)
if run_line:
    run_root = Path(run_line.split(': ', 1)[1].strip())
    print("Export directory:", run_root / 'nexsafe_export')

## Manual deployment

After successful completion, use Kaggle's notebook output browser to download the checkpoint, its `.nexsafe.json` sidecar, and the JSON report. Copy both model files into NexSafe's mounted model directory, set `MODEL_WEIGHTS=/models/uvh26-yolo11n.pt`, and restart the API. Confirm `/api/model` reports the checkpoint version/classes and the actual validation metrics.

The detector is trained only on the subset described in its report; UVH-26 has no held-out test split. The notebook will not claim test performance. Temporal accident and structured-risk checkpoints require separately verified labels/manifests/CSV files and should be trained using the existing scripts only after their source semantics, license, and leakage-safe group splits are checked.